In [1]:
# notebooks/03_Test.ipynb

# 1. Environment Initialization
import os
import shutil
import sys
import random
import itertools
import numpy as np
import pandas as pd
import torch
from torch.amp import autocast
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Install the pre-compiled files instantly from Drive
print("Installing mamba_wheels from drive...")
!pip install /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/data/mamba_wheels/*.whl
print("Mamba_wheels installed.")

Installing mamba_wheels from drive...
Processing ./drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/data/mamba_wheels/causal_conv1d-1.7.0-cp313-cp313-linux_x86_64.whl
Processing ./drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/data/mamba_wheels/mamba_ssm-2.3.0-cp313-cp313-linux_x86_64.whl
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 9.2 MB/s eta 0:00:00
Mamba_wheels installed.


In [3]:
!pip install -q monai
from monai.inferers import sliding_window_inference

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 30.5 MB/s eta 0:00:00


In [4]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

PROJECT_ROOT = "/content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor"
sys.path.append(PROJECT_ROOT)

import src.config as config
from src.dataset import get_test_dataloader
from src.metrics import SegmentationMetrics
from src.models.mamba_backbone import MambaBackbone, SharedDeepMambaBackbone
from src.models.fusion import PresenceAwareCrossModalFusion
from src.models.decoder import SegmentationDecoder3D, AuxiliaryDecoder3D

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
DATASET_ZIP = "/content/drive/MyDrive/ML-Datasets/BraTS2020_TrainingData_ABC.zip"
LOCAL_EXTRACT_DIR = "/content/MICCAI_BraTS2020_TrainingData_ABC"
LOCAL_DATA_DIR = LOCAL_EXTRACT_DIR

# Verify archive existence immediately before runtime allocation
assert os.path.exists(DATASET_ZIP), f"Dataset archive not found: {DATASET_ZIP}"

# Robust extraction guard: triggers if directory does not exist or is completely empty
if not os.path.exists(LOCAL_DATA_DIR) or len(os.listdir(LOCAL_DATA_DIR)) == 0:
    print(f"Extracting preprocessed dataset to local runtime storage: {LOCAL_EXTRACT_DIR}...")
    os.makedirs(LOCAL_EXTRACT_DIR, exist_ok=True)
    shutil.unpack_archive(DATASET_ZIP, LOCAL_EXTRACT_DIR, "zip")
    print("Extraction complete.")
else:
    print("Existing local preprocessed dataset detected. Skipping extraction.")

Extracting preprocessed dataset to local runtime storage: /content/MICCAI_BraTS2020_TrainingData_ABC...
Extraction complete.


In [6]:
import importlib

importlib.reload(config)

import src.engine as engine
importlib.reload(engine)

import src.models.mamba_backbone as mbb
importlib.reload(mbb)

import src.models.fusion as mf
importlib.reload(mf)

import src.models.decoder as md
importlib.reload(md)

import src.losses as sl
importlib.reload(sl)

import src.transforms as tf
importlib.reload(tf)

print("Reloaded all src files.")

Reloaded all src files.


In [7]:
# 2. Pipeline Dataset Loaders Construction
test_loader = get_test_dataloader(batch_size = 4) # defauld set to batch_size=config.BATCH_SIZE

# 3. Structural Module Instantiations & Checkpoint Loading
backbone = MambaBackbone(embed_dim=config.EMBED_DIM).to(device)
fusion = PresenceAwareCrossModalFusion(embed_dim=config.EMBED_DIM).to(device)
shared_backbone = SharedDeepMambaBackbone(embed_dim=config.EMBED_DIM).to(device)
decoder = SegmentationDecoder3D(embed_dim=config.EMBED_DIM, out_channels=config.NUM_SEG_CLASSES).to(device)
aux_decoder = AuxiliaryDecoder3D(embed_dim=config.EMBED_DIM, out_channels=config.NUM_SEG_CLASSES).to(device)

best_seg_path = os.path.join(config.CHECKPOINT_DIR, "best_seg.pth")
assert os.path.exists(best_seg_path), f"Checkpoint not found: {best_seg_path}"

checkpoint = torch.load(best_seg_path, map_location=device)
backbone.load_state_dict(checkpoint["backbone_state"])
fusion.load_state_dict(checkpoint["fusion_state"])
shared_backbone.load_state_dict(checkpoint["shared_backbone_state"])
decoder.load_state_dict(checkpoint["decoder_state"])
aux_decoder.load_state_dict(checkpoint["aux_decoder_state"])

backbone.eval()
fusion.eval()
shared_backbone.eval()
decoder.eval()

print("Checkpoint loaded.")

Checkpoint loaded.


In [8]:
# 4. Define 15 Modality Combinations
# 0: T1, 1: T1ce, 2: T2, 3: FLAIR
mod_idx = [0, 1, 2, 3]
mod_names = ["T1", "T1ce", "T2", "FLAIR"]

combinations = []
for r in range(1, 5):
    combinations.extend(list(itertools.combinations(mod_idx, r)))

# --- Resumption and Directory Setup ---
csv_out_path = os.path.join(config.RESULT_DIR, "modality_dropout_results.csv")

completed_combinations = []
if os.path.exists(csv_out_path):
    existing_df = pd.read_csv(csv_out_path)
    if "Present Modalities" in existing_df.columns:
        completed_combinations = existing_df["Present Modalities"].tolist()
        print(f"[*] Found existing results. Resuming... ({len(completed_combinations)}/15 completed)")
# --------------------------------------

results = []

In [9]:
# 5. Evaluation Loop across 15 settings
print(f"Starting multi-modality evaluation across {len(combinations)} settings...")

with torch.no_grad():
    for comb in combinations:
        comb_names = "+".join([mod_names[i] for i in comb])

        # --- Skip if already processed ---
        if comb_names in completed_combinations:
            print(f"Skipping {comb_names} (already completed)...")
            continue
        # ---------------------------------

        print(f"\n--- Testing Modalities: {comb_names} ---")

        keep_mask = torch.zeros(4, device=device)
        keep_mask[list(comb)] = 1.0

        seg_tracker = SegmentationMetrics()

        for batch in tqdm(test_loader, desc=f"Eval {comb_names}", leave=False):
            images = batch["image"].to(device)
            seg_targets = batch["label"].to(device)
            B_current = images.size(0)
            batch_seg_logits = []

            for b in range(B_current):
                single_img = images[b:b+1]

                def evaluation_predictor(patch_images):
                    # 1. Apply dropout to the raw input images FIRST
                    processed_images = patch_images.clone()
                    for i in range(4):
                        if keep_mask[i] == 0.0:
                            processed_images[:, i, :, :, :] = 0.0

                    # 2. Backbone now processes genuinely incomplete inputs
                    modality_tokens, spatial_shape, skip_features, _ = backbone(processed_images)

                    # 3. Format tokens for the fusion module as required
                    processed_modality_tokens = []
                    for i in range(4):
                        if keep_mask[i] == 1.0:
                            processed_modality_tokens.append(modality_tokens[i])
                        else:
                            processed_modality_tokens.append(torch.zeros_like(modality_tokens[i]))

                    fused_tokens = fusion(processed_modality_tokens, processed_images)
                    latent_tokens = shared_backbone(fused_tokens)
                    seg_logits = decoder(latent_tokens, spatial_shape, skip_features)

                    return seg_logits

                with autocast(device_type=device.type, enabled=(device.type == "cuda")):
                    seg_logits = sliding_window_inference(
                        inputs=single_img,
                        roi_size=config.PATCH_SIZE,
                        sw_batch_size=24,
                        predictor=evaluation_predictor,
                        overlap=0.5,
                        mode="gaussian"
                    )
                batch_seg_logits.append(seg_logits)

            seg_logits = torch.cat(batch_seg_logits, dim=0)
            seg_preds = torch.argmax(seg_logits, dim=1, keepdim=True)
            seg_tracker.update(seg_preds, seg_targets, run_hd=False)

        metrics = seg_tracker.compute(run_hd=False)
        mean_dice = (metrics["dice_WT"] + metrics["dice_TC"] + metrics["dice_ET"]) / 3.0

        current_result = {
            "Missing Modalities": "+".join([mod_names[i] for i in mod_idx if i not in comb]) or "None",
            "Present Modalities": comb_names,
            "Dice WT": round(metrics["dice_WT"], 4),
            "Dice TC": round(metrics["dice_TC"], 4),
            "Dice ET": round(metrics["dice_ET"], 4),
            "Mean Dice": round(mean_dice, 4),
            "HD95 WT": round(metrics["hd95_WT"], 4) if "hd95_WT" in metrics else "N/A",
            "HD95 TC": round(metrics["hd95_TC"], 4) if "hd95_TC" in metrics else "N/A",
            "HD95 ET": round(metrics["hd95_ET"], 4) if "hd95_ET" in metrics else "N/A"
        }

        results.append(current_result)
        print(f"Mean Dice: {mean_dice:.4f} | WT: {metrics['dice_WT']:.4f}, TC: {metrics['dice_TC']:.4f}, ET: {metrics['dice_ET']:.4f}")

        # --- Incremental Save to CSV ---
        current_df = pd.DataFrame([current_result])
        if not os.path.exists(csv_out_path):
            current_df.to_csv(csv_out_path, index=False)
        else:
            current_df.to_csv(csv_out_path, mode='a', header=False, index=False)
        # -------------------------------

Starting multi-modality evaluation across 15 settings...

--- Testing Modalities: T1 ---


Mean Dice: 0.5296 | WT: 0.6219, TC: 0.5768, ET: 0.3902

--- Testing Modalities: T1ce ---


Mean Dice: 0.7266 | WT: 0.6582, TC: 0.7471, ET: 0.7746

--- Testing Modalities: T2 ---


Mean Dice: 0.6528 | WT: 0.8065, TC: 0.6644, ET: 0.4876

--- Testing Modalities: FLAIR ---


Mean Dice: 0.6213 | WT: 0.8049, TC: 0.6265, ET: 0.4324

--- Testing Modalities: T1+T1ce ---


...

Mean Dice: 0.8329 | WT: 0.8377, TC: 0.8236, ET: 0.8376

--- Testing Modalities: T1+T1ce+FLAIR ---


Mean Dice: 0.8475 | WT: 0.8698, TC: 0.8339, ET: 0.8389

--- Testing Modalities: T1+T2+FLAIR ---


Mean Dice: 0.7032 | WT: 0.8820, TC: 0.7051, ET: 0.5223

--- Testing Modalities: T1ce+T2+FLAIR ---


Mean Dice: 0.8587 | WT: 0.8853, TC: 0.8474, ET: 0.8436

--- Testing Modalities: T1+T1ce+T2+FLAIR ---


Mean Dice: 0.8585 | WT: 0.8861, TC: 0.8470, ET: 0.8424


In [10]:
# 6. Display Results
print("\nTesting finished. Loading final combined results:")
if os.path.exists(csv_out_path):
    df_results = pd.read_csv(csv_out_path)
    display(df_results)
else:
    print("No results found.")


Testing finished. Loading final combined results:


,Missing Modalities,Present Modalities,Dice WT,Dice TC,Dice ET,Mean Dice,HD95 WT,HD95 TC,HD95 ET
0,T1ce+T2+FLAIR,T1,0.6219,0.5768,0.3902,0.5296,0.0,0.0,0.0
1,T1+T2+FLAIR,T1ce,0.6582,0.7471,0.7746,0.7266,0.0,0.0,0.0
2,T1+T1ce+FLAIR,T2,0.8065,0.6644,0.4876,0.6528,0.0,0.0,0.0
3,T1+T1ce+T2,FLAIR,0.8049,0.6265,0.4324,0.6213,0.0,0.0,0.0
4,T2+FLAIR,T1+T1ce,0.6942,0.8038,0.8134,0.7705,0.0,0.0,0.0
5,T1ce+FLAIR,T1+T2,0.8287,0.6741,0.5034,0.6687,0.0,0.0,0.0
6,T1ce+T2,T1+FLAIR,0.8665,0.6799,0.4783,0.6749,0.0,0.0,0.0
7,T1+FLAIR,T1ce+T2,0.8314,0.8277,0.8310,0.8300,0.0,0.0,0.0
8,T1+T2,T1ce+FLAIR,0.8648,0.8242,0.8322,0.8404,0.0,0.0,0.0
9,T1+T1ce,T2+FLAIR,0.8713,0.6937,0.5149,0.6933,0.0,0.0,0.0


In [11]:
# 6. Display Results
print("\nTesting finished. Loading final combined results:")
if os.path.exists(csv_out_path):
    import numpy as np
    import pandas as pd

    df_results = pd.read_csv(csv_out_path)

    mean_row = {}
    for col in df_results.columns:
        if col == "Missing Modalities":
            mean_row[col] = "OVERALL MEAN"
        elif col == "Present Modalities":
            mean_row[col] = "-"
        else:
            col_numeric = pd.to_numeric(df_results[col], errors='coerce')
            if not col_numeric.isna().all():
                mean_row[col] = round(col_numeric.mean(), 4)
            else:
                mean_row[col] = "N/A"

    df_results.loc[len(df_results)] = mean_row
    display(df_results)
else:
    print("No results found.")


Testing finished. Loading final combined results:


,Missing Modalities,Present Modalities,Dice WT,Dice TC,Dice ET,Mean Dice,HD95 WT,HD95 TC,HD95 ET
0,T1ce+T2+FLAIR,T1,0.6219,0.5768,0.3902,0.5296,0.0,0.0,0.0
1,T1+T2+FLAIR,T1ce,0.6582,0.7471,0.7746,0.7266,0.0,0.0,0.0
2,T1+T1ce+FLAIR,T2,0.8065,0.6644,0.4876,0.6528,0.0,0.0,0.0
3,T1+T1ce+T2,FLAIR,0.8049,0.6265,0.4324,0.6213,0.0,0.0,0.0
4,T2+FLAIR,T1+T1ce,0.6942,0.8038,0.8134,0.7705,0.0,0.0,0.0
5,T1ce+FLAIR,T1+T2,0.8287,0.6741,0.5034,0.6687,0.0,0.0,0.0
6,T1ce+T2,T1+FLAIR,0.8665,0.6799,0.4783,0.6749,0.0,0.0,0.0
7,T1+FLAIR,T1ce+T2,0.8314,0.8277,0.8310,0.8300,0.0,0.0,0.0
8,T1+T2,T1ce+FLAIR,0.8648,0.8242,0.8322,0.8404,0.0,0.0,0.0
9,T1+T1ce,T2+FLAIR,0.8713,0.6937,0.5149,0.6933,0.0,0.0,0.0
